# MiniGrid Benchmark Runner
Notebook para executar benchmark_minigrid em ambiente local, Colab ou Kaggle.

## 1 - Instalações e Configurações Diversas

Basta executar tudo desta seção. Não precisa alterar nada.

In [ ]:
try:
    import kaggle_secrets
    EXECUTION_ENV = "kaggle"
except ImportError:
    try:
        import google.colab
        EXECUTION_ENV = "colab"
    except ImportError:
        EXECUTION_ENV = "local"

print(f"Execution environment: {EXECUTION_ENV}")

In [ ]:
if EXECUTION_ENV in ["colab", "kaggle"]:
    !pip install -q minigrid==3.1.0
    !pip install -q langchain-openai>=0.0.1 langchain-deepseek>=0.0.1 langchain-huggingface==1.2.2
    !pip install -q transformers>=5.5.4 bitsandbytes

In [ ]:
import sys
import os

if EXECUTION_ENV == "colab":
    repo_path = os.path.join(os.getcwd(), "minigrid_benchmark")
elif EXECUTION_ENV == "kaggle":
    repo_path = "/kaggle/working/minigrid_benchmark"
else:
    cwd = os.getcwd()
    candidates = [cwd, os.path.abspath(os.path.join(cwd, ".."))]
    repo_path = None
    for candidate in candidates:
        if os.path.exists(os.path.join(candidate, "src", "benchmark_minigrid.py")):
            repo_path = candidate
            break
    if repo_path is None:
        repo_path = cwd

if EXECUTION_ENV in ["colab", "kaggle"] and not os.path.exists(repo_path):
    !git clone https://github.com/pablo-sampaio/minigrid_benchmark.git {repo_path}

src_path = os.path.join(repo_path, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

print(f"repo_path={repo_path}")
print(f"src_path={src_path}")

In [ ]:
from run_full_benchmark_minigrid_helpers import (
    MODEL_OPTIONS,
    configure_results_dir,
    create_model_selector_widgets,
    resolve_api_key,
    resume_from_previous_results_folder,
    zip_results_for_export,
    )
from benchmark_minigrid import run_benchmark_minigrid

In [ ]:
results_dir = configure_results_dir(EXECUTION_ENV, repo_path)
print(f"results_dir={results_dir}")

## 2 - Defina Aqui o Modelo

In [ ]:
from IPython.display import display

provider_dd, model_dd, model_selection = create_model_selector_widgets(MODEL_OPTIONS)
display(provider_dd, model_dd)

In [ ]:
experiment_name = None

# caso queria continuar, indicar o nome da pasta
#results_folder_name = 'benchmark_openai_gpt-5.4_2026-06-20_08h27min'

## 3 - Execução dos Testes do Benchark

In [ ]:
PROVIDER = model_selection["provider"]
MODEL_ID = model_selection["model_id"]
QUANTIZATION = model_selection["quantization"]

API_KEY = resolve_api_key(PROVIDER, execution_env=EXECUTION_ENV)

In [ ]:
if EXECUTION_ENV == "kaggle":
    experiment_name = resume_from_previous_results_folder(
        repo_path=repo_path,
        next_results_dir=results_dir,
        provider=PROVIDER,
        model_id=MODEL_ID,
        experiment_name_to_resume=experiment_name,
    )

In [ ]:
#experiment_name

In [ ]:
final_results, summary_path = run_benchmark_minigrid(
    provider=PROVIDER,
    model_id=MODEL_ID,
    api_key=API_KEY,
    results_base_dir=results_dir,
    results_folder_name=experiment_name,
    quantization=QUANTIZATION if PROVIDER == "hf" else None,
    verbose=True,
)

print('Saved summary to:', summary_path)

In [ ]:
print("Arquivo JSON com o sumário dos resultados:", summary_path)

In [ ]:
zip_path = zip_results_for_export(EXECUTION_ENV, summary_path)
if zip_path:
    print("Arquivo compactado com todos os resultados:", zip_path)